# Preprocesamiento para el modelo — House Pricing

Este notebook transforma las 30 variables seleccionadas en el análisis exploratorio mediante un pipeline de scikit-learn con transformadores de `feature-engine`. Genera los datasets de entrada para el notebook de construcción de modelos.

## 1. Librerías

Importación de las librerías para manipulación de datos, transformación de variables, construcción del pipeline y persistencia de resultados.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from feature_engine.encoding import MeanEncoder
from feature_engine.imputation import AddMissingIndicator, CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

## 2. Configuración y carga de datos

Lectura de `pipeline_config.yaml` y de los conjuntos `train` y `test`. Las rutas de los datos y la lista de variables provienen de ese archivo; las rutas se resuelven desde `PROJ_ROOT`.

In [2]:
PROJ_ROOT = Path("..")
CONFIG_PATH = PROJ_ROOT / "pipeline_config.yaml"

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = yaml.safe_load(f)

TARGET = config["TARGET"]
MODEL_FEATURES = config["MODEL_FEATURES"]
RAW_DIR = PROJ_ROOT / config["DATA"]["RAW_DIR"]

train = pd.read_csv(RAW_DIR / config["DATA"]["TRAIN_FILE"])
test = pd.read_csv(RAW_DIR / config["DATA"]["TEST_FILE"])

print(f"train: {train.shape}")
print(f"test:  {test.shape}")

train: (1460, 81)
test:  (1459, 80)


## 3. Grupos de variables

Clasificación de las variables según el tratamiento que reciben:

- **Continuas:** grupo definido en la selección de features.
- **Discretas:** grupo definido en la selección de features, excepto `MSSubClass`.
- **Ordinales:** escalas de calidad con orden natural.
- **Nominales:** categóricas sin orden, más `MSSubClass`, que es un código de tipo de vivienda y no una cantidad.

In [3]:
CONTINUAS = config["FEATURES"]["CONTINUAS"]
DISCRETAS = [col for col in config["FEATURES"]["DISCRETAS"] if col != "MSSubClass"]
ORDINALES = ["ExterQual", "BsmtQual", "KitchenQual", "FireplaceQu", "HeatingQC"]
NOMINALES = [col for col in config["FEATURES"]["CATEGORICAS"] if col not in ORDINALES] + ["MSSubClass"]

print(f"Continuas ({len(CONTINUAS)}): {CONTINUAS}")
print(f"Discretas ({len(DISCRETAS)}): {DISCRETAS}")
print(f"Ordinales ({len(ORDINALES)}): {ORDINALES}")
print(f"Nominales ({len(NOMINALES)}): {NOMINALES}")

Continuas (10): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']
Discretas (9): ['OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces', 'OverallCond', 'HalfBath', 'BedroomAbvGr', 'BsmtFullBath']
Ordinales (5): ['ExterQual', 'BsmtQual', 'KitchenQual', 'FireplaceQu', 'HeatingQC']
Nominales (6): ['Neighborhood', 'GarageFinish', 'Foundation', 'GarageType', 'BsmtFinType1', 'MSSubClass']


## 4. Transformación del target y división de datos

`SalePrice` se transforma con `log1p` para reducir su asimetría; las predicciones se regresan a dólares con `np.expm1`. Después se separa un 20% para validación. El pipeline se ajusta solo con el 80% restante, ya que `MeanEncoder` utiliza el target y ajustarlo con todos los datos filtraría información a la validación.

In [4]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

X = train[MODEL_FEATURES]
y = np.log1p(train[TARGET])
X_test = test[MODEL_FEATURES]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"X_train: {X_train.shape}")
print(f"X_valid: {X_valid.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (1168, 30)
X_valid: (292, 30)
X_test:  (1459, 30)


## 5. Imputación de variables

### 5.1 Variables numéricas

Se agrega una columna binaria (`_na`) para `LotFrontage`, `GarageYrBlt` y `MasVnrArea`, que conserva la información de que el dato no existía. Después se imputan todas las variables continuas y discretas con la mediana. La imputación incluye las variables que solo presentan faltantes en `test`.

In [5]:
VARIABLES_INDICADOR = ["LotFrontage", "GarageYrBlt", "MasVnrArea"]

missing_indicator = AddMissingIndicator(variables=VARIABLES_INDICADOR)
numeric_imputer = MeanMedianImputer(
    imputation_method="median", variables=CONTINUAS + DISCRETAS
)

### 5.2 Variables categóricas

En `BsmtQual`, `BsmtFinType1`, `GarageType`, `GarageFinish` y `FireplaceQu`, un valor vacío indica que la vivienda no tiene esa característica, según `data_description.txt`. Estas variables se imputan con la categoría `Missing` para conservar esa información.

Las demás variables categóricas se imputan con la moda (categoría más frecuente en `X_train`). `MSSubClass` se incluye con `ignore_format=True` porque está almacenada como número.

In [6]:
VARIABLES_SIN_CARACTERISTICA = ["BsmtQual", "BsmtFinType1", "GarageType", "GarageFinish", "FireplaceQu"]
VARIABLES_MODA = [
    col for col in ORDINALES + NOMINALES if col not in VARIABLES_SIN_CARACTERISTICA
]

missing_imputer = CategoricalImputer(
    imputation_method="missing", fill_value="Missing", variables=VARIABLES_SIN_CARACTERISTICA
)
frequent_imputer = CategoricalImputer(
    imputation_method="frequent", variables=VARIABLES_MODA, ignore_format=True
)

## 6. Codificación de variables categóricas

### 6.1 Variables ordinales

Las escalas de calidad se codifican según su orden natural: `Missing` < `Po` < `Fa` < `TA` < `Gd` < `Ex`. `Missing`, que indica que la vivienda no tiene la característica, ocupa el valor más bajo. `feature-engine` no admite un orden definido manualmente, por lo que se utiliza `OrdinalEncoder` de scikit-learn con `SklearnTransformerWrapper`.

In [7]:
ESCALA_CALIDAD = ["Missing", "Po", "Fa", "TA", "Gd", "Ex"]

ordinal_encoder = SklearnTransformerWrapper(
    transformer=OrdinalEncoder(categories=[ESCALA_CALIDAD] * len(ORDINALES)),
    variables=ORDINALES,
)

### 6.2 Variables nominales

Cada categoría se reemplaza por el promedio del target. Con `smoothing="auto"`, las categorías con pocas observaciones se acercan al promedio global, lo que reduce el sobreajuste. Con `unseen="encode"`, las categorías no vistas en `X_train`, como el código 150 de `MSSubClass` en `test`, reciben el promedio global.

In [8]:
mean_encoder = MeanEncoder(
    variables=NOMINALES, smoothing="auto", unseen="encode", ignore_format=True
)

## 7. Tratamiento de outliers

Los valores de las variables continuas se limitan a los extremos Q1 − 1.5·IQR y Q3 + 1.5·IQR calculados en `X_train`. No se eliminan filas.

In [9]:
winsorizer = Winsorizer(capping_method="iqr", tail="both", fold=1.5, variables=CONTINUAS)

## 8. Transformación de variables continuas

Se aplica Yeo-Johnson a las seis variables continuas con asimetría mayor a 0.75. Este método acepta ceros y estima la potencia óptima de cada variable en `X_train`.

In [10]:
VARIABLES_ASIMETRICAS = [
    "MasVnrArea",
    "LotFrontage",
    "BsmtFinSF1",
    "TotalBsmtSF",
    "1stFlrSF",
    "GrLivArea",
]

yeo_johnson = YeoJohnsonTransformer(variables=VARIABLES_ASIMETRICAS)

## 9. Escalado de variables

`StandardScaler` lleva todas las columnas finales a media 0 y desviación estándar 1. Se utiliza con `SklearnTransformerWrapper` para conservar el formato de `DataFrame`.

In [11]:
scaler = SklearnTransformerWrapper(transformer=StandardScaler())

## 10. Pipeline de preprocesamiento

Integración de los pasos anteriores en un `Pipeline` de scikit-learn, ajustado únicamente con `X_train` y `y_train`.

In [12]:
preprocessing_pipeline = Pipeline([
    ("missing_indicator", missing_indicator),
    ("numeric_imputer", numeric_imputer),
    ("missing_imputer", missing_imputer),
    ("frequent_imputer", frequent_imputer),
    ("ordinal_encoder", ordinal_encoder),
    ("mean_encoder", mean_encoder),
    ("winsorizer", winsorizer),
    ("yeo_johnson", yeo_johnson),
    ("scaler", scaler),
])

preprocessing_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('missing_indicator', ...), ('numeric_imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,variables,"['LotFrontage', 'GarageYrBlt', ...]"
,missing_only,True
Name,Type,Value
feature_names_in_,list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"


## 11. Generación de datasets

Aplicación del pipeline a los tres conjuntos. `train` y `valid` incluyen `SalePrice` en escala logarítmica; `test` incluye `Id` y no tiene target.

In [13]:
train_processed = preprocessing_pipeline.transform(X_train).assign(**{TARGET: y_train})
valid_processed = preprocessing_pipeline.transform(X_valid).assign(**{TARGET: y_valid})
test_processed = preprocessing_pipeline.transform(X_test)
test_processed.insert(0, "Id", test["Id"])

print(f"train_processed: {train_processed.shape}")
print(f"valid_processed: {valid_processed.shape}")
print(f"test_processed:  {test_processed.shape}")
train_processed.head()

train_processed: (1168, 34)
valid_processed: (292, 34)
test_processed:  (1459, 34)


,GrLivArea,GarageArea,TotalBsmtSF,1stFlrSF,YearBuilt,YearRemodAdd,GarageYrBlt,MasVnrArea,BsmtFinSF1,LotFrontage,...,GarageFinish,FireplaceQu,Foundation,GarageType,BsmtFinType1,HeatingQC,LotFrontage_na,GarageYrBlt_na,MasVnrArea_na,SalePrice
254,-0.322407,-0.879058,0.663883,0.528730,-0.456671,-1.346063,-0.904889,-0.839920,0.972365,0.041733,...,0.598398,-1.016241,-0.723701,0.582043,-0.938708,-1.168292,-0.477683,-0.240772,-0.071858,11.884496
1066,0.237782,-0.459359,-0.624720,-1.052262,0.719207,0.439214,0.593655,-0.839920,-1.356108,-0.596909,...,0.598398,0.639226,1.076647,0.582043,-0.222227,-0.132176,-0.477683,-0.240772,-0.071858,12.089544
638,-1.782809,-2.313843,-0.632449,-1.063812,-1.991844,-1.683818,0.052514,-0.839920,-1.356108,-0.131767,...,-2.266416,-1.016241,-0.723701,-2.440363,-0.222227,-0.132176,-0.477683,4.153312,-0.071858,11.350418
799,0.620380,-1.142590,-0.800713,-0.413089,-1.109936,-1.683818,-1.654161,1.262816,0.661285,-0.538559,...,-0.819726,0.639226,-1.281113,-1.089553,-0.441786,0.903941,-0.477683,-0.240772,-0.071858,12.072547
380,0.475013,-0.810735,-0.048167,-0.270988,-1.534558,-1.683818,-2.278554,-0.839920,0.138594,-1.125025,...,-0.819726,1.191048,-1.281113,-1.089553,-0.853361,-1.168292,-0.477683,-0.240772,-0.071858,11.751950


## 12. Guardado de resultados

Se guardan los datasets en `data/processed/` y el pipeline ajustado en `models/`. Sus rutas y los parámetros de la división se agregan a `pipeline_config.yaml` para el notebook de modelos.

In [14]:
PROCESSED_DIR = "data/processed"
MODELS_DIR = "models"
PROCESSED_FILES = {
    "TRAIN_FILE": "train_processed.csv",
    "VALID_FILE": "valid_processed.csv",
    "TEST_FILE": "test_processed.csv",
}
PIPELINE_FILE = "preprocessing_pipeline.joblib"

processed_path = PROJ_ROOT / PROCESSED_DIR
train_processed.to_csv(processed_path / PROCESSED_FILES["TRAIN_FILE"], index=False)
valid_processed.to_csv(processed_path / PROCESSED_FILES["VALID_FILE"], index=False)
test_processed.to_csv(processed_path / PROCESSED_FILES["TEST_FILE"], index=False)

joblib.dump(preprocessing_pipeline, PROJ_ROOT / MODELS_DIR / PIPELINE_FILE)

config["SPLIT"] = {"TEST_SIZE": TEST_SIZE, "RANDOM_STATE": RANDOM_STATE}
config["TARGET_TRANSFORM"] = "log1p"
config["PROCESSED"] = {"DIR": PROCESSED_DIR, **PROCESSED_FILES}
config["PIPELINE_PATH"] = f"{MODELS_DIR}/{PIPELINE_FILE}"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print("Datasets, pipeline y configuración guardados.")

Datasets, pipeline y configuración guardados.
